# 🚀 Модуль 7. Урок 1. Фреймворк A/B-тестирования: MDE, статистическая мощность (Power) и Sample Size

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Enlyl/data-analyst-2027/blob/main/course/module_07_ab_testing/lesson_01_ab_testing_framework_and_mde.ipynb)

## 📌 Бизнес-контекст и методология
Дизайн эксперимента начинается с расчета необходимого объема выборки:
* $\alpha$ (False Positive Rate / Ошибка I рода, 5%)
* $\beta$ (False Negative Rate / Ошибка II рода, 20% -> Мощность $1 - \beta = 80\%$)
* MDE (Minimum Detectable Effect) — минимальный значимый для бизнеса эффект.

In [1]:
# 🚀 Универсальная инициализация окружения и данных
import os
import urllib.request
import sqlite3
import pandas as pd
import numpy as np

def ensure_dataset(filename, subfolder=""):
    """Автоматически находит датасет локально или скачивает с GitHub репозитория курса."""
    if os.path.exists(filename):
        return filename
    candidates = [
        os.path.join("course", "datasets", subfolder, filename),
        os.path.join("course", "datasets", filename),
        os.path.join("..", "datasets", subfolder, filename),
        os.path.join("..", "datasets", filename),
        os.path.join("..", "..", "course", "datasets", subfolder, filename),
        os.path.join("datasets", subfolder, filename),
        os.path.join("datasets", filename),
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    raw_base = "https://raw.githubusercontent.com/Enlyl/data-analyst-2027/main/course/datasets"
    url = f"{raw_base}/{subfolder}/{filename}" if subfolder else f"{raw_base}/{filename}"
    print(f"📥 Загрузка {filename} из репозитория курса...")
    urllib.request.urlretrieve(url, filename)
    print(f"✅ Датасет {filename} успешно готов!")
    return filename

# Загрузка ключевых датасетов
customers_csv = ensure_dataset("customers.csv", subfolder="raw")
orders_csv = ensure_dataset("orders.csv", subfolder="raw")
saas_csv = ensure_dataset("saas_users.csv", subfolder="raw")
print("🟢 Окружение готово к анализу данных.")


🟢 Окружение готово к анализу данных.


## 🔍 Разбор механики и демонстрация на реальных данных

In [2]:
import numpy as np
from scipy import stats

def calculate_sample_size(p_base, mde_rel, alpha=0.05, power=0.80):
    p_treat = p_base * (1 + mde_rel)
    z_alpha = stats.norm.ppf(1 - alpha / 2)
    z_beta = stats.norm.ppf(power)
    
    p_pool = (p_base + p_treat) / 2
    var_pooled = 2 * p_pool * (1 - p_pool)
    
    n_per_group = ((z_alpha + z_beta) ** 2 * var_pooled) / ((p_treat - p_base) ** 2)
    return int(np.ceil(n_per_group))

base_cr = 0.05
desired_mde = 0.10
needed_n = calculate_sample_size(base_cr, desired_mde)

print(f"🎯 Необходимый размер выборки на 1 группу: {needed_n:,} пользователей")
print(f"👥 Всего для A/B теста (A + B):            {needed_n * 2:,} пользователей")


🎯 Необходимый размер выборки на 1 группу: 31,235 пользователей
👥 Всего для A/B теста (A + B):            62,470 пользователей


### 🎯 Практическое задание 1: Оценка зависимости выборки от MDE
Рассчитайте необходимый размер выборки для MDE = 5% (0.05).

In [3]:
### ВАШ КОД ЗДЕСЬ
smaller_mde = 0.05
n_for_small_mde = calculate_sample_size(base_cr, smaller_mde)

print(f"Размер выборки при MDE 5%: {n_for_small_mde:,} пользователей")
assert n_for_small_mde > needed_n * 3, "Уменьшение MDE в 2 раза требует ~4-кратного роста выборки!"
print("✅ Тест успешно пройден: квадратичная зависимость выборки от MDE подтверждена!")


Размер выборки при MDE 5%: 122,125 пользователей
✅ Тест успешно пройден: квадратичная зависимость выборки от MDE подтверждена!


### 💡 Золотой закон A/B тестирования
Размер выборки обратно пропорционален квадрату MDE. Обнаружение микроэффектов требует колоссального трафика.